## Import Library

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

## Load Data

In [6]:
#train = pd.read_csv('sales_train.csv', parse_dates=['date'])
#test = pd.read_csv('sales_test.csv', parse_dates=['date'])
#ss = pd.read_csv('solution.csv')
#inventory = pd.read_csv('inventory.csv')
#weights = pd.read_csv('test_weights.csv')
calendar = pd.read_csv('calendar.csv', parse_dates=['date'])

#### Check data before we do anything
七个warehouse，五个国家
930行有holiday
166个有holiday没有名字

In [7]:
warehouses = calendar["warehouse"].unique()
print(warehouses)
print(calendar["holiday_name"].count())
print(calendar["holiday_name"].unique())
print((calendar["winter_school_holidays"] == 1).sum())
print((calendar["school_holidays"] == 1).sum())
print(calendar[(calendar["holiday"] == 1) & (pd.isna(calendar["holiday_name"]))])

['Frankfurt_1' 'Prague_2' 'Brno_1' 'Munich_1' 'Prague_3' 'Prague_1'
 'Budapest_1']
930
[nan 'Den boje za svobodu a demokracii' 'Good Friday' 'Easter Monday'
 '2nd Christmas Day' 'Cyrila a Metodej' 'International womens day'
 'Den ceske statnosti' 'Den osvobozeni' 'New Years Day' 'Whit sunday'
 'Memorial Day of the Republic' 'Independent Hungary Day' 'Labour Day'
 'Memorial Day for the Victims of the Holocaust' 'Reformation Day'
 'Den vzniku samostatneho ceskoslovenskeho statu' 'Ascension day'
 'Corpus Christi' 'Jan Hus' 'Assumption of the Virgin Mary' 'Epiphany'
 'Christmas Eve' 'Memorial day of the 1956 Revolution'
 'Memorial Day for the Martyrs of Arad' 'Day of National Unity'
 '1st Christmas Day' 'Whit monday' 'German Unity Day'
 'State Foundation Day' 'All Saints Day' 'Hungary National Day Holiday'
 'Christmas Holiday'
 'Memorial Day for the Victims of the Communist Dictatorships'
 'Peace Festival in Augsburg' 'National Defense Day'
 '1848 Revolution Memorial Day (Extra holiday)' "

## 补充节日

In [8]:
# Holidays get from https://www.holidays-info.com/
# https://www.holidays-info.com/czech-republic/calendar/prague/2024/

# 蒋宁致你记得看看我们需不需要：只留下需要的几列特征
calendar = calendar[['warehouse', 'date', 'holiday', 'holiday_name']]

czech_holiday = [  # Prague
    (['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020', '03/27/2016', '04/16/2017', '04/01/2018', '04/21/2019'], 'Easter Day'), 
    (['04/01/2024', '04/10/2023', '04/18/2022', '04/05/2021', '04/13/2020', '03/28/2016', '04/17/2017', '04/02/2018', '04/22/2019'], 'Easter Monday'),
    (['05/12/2024', '05/10/2020', '05/09/2021', '05/08/2022', '05/14/2023', '05/08/2016', '05/14/2017', '05/13/2018', '05/12/2019'], "Mother's Day"), 
    (['03/25/2016', '04/14/2017', '03/30/2018', '04/20/2019', '04/11/2020', '04/03/2021', '04/16/2022', '04/08/2023'], 'Good Friday'),
]

brno_holiday = [  # Brno
    (['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020', '03/27/2016', '04/16/2017', '04/01/2018', '04/21/2019'], 'Easter Day'), 
    (['04/01/2024', '04/10/2023', '04/18/2022', '04/05/2021', '04/13/2020', '03/28/2016', '04/17/2017', '04/02/2018', '04/22/2019'], 'Easter Monday'),
    (['05/12/2024', '05/10/2020', '05/09/2021', '05/08/2022', '05/14/2023', '05/08/2016', '05/14/2017', '05/13/2018', '05/12/2019'], "Mother's Day"),
    (['03/25/2016', '04/14/2017', '03/30/2018', '04/20/2019', '04/11/2020', '04/03/2021', '04/16/2022', '04/08/2023'], 'Good Friday'),
]

budapest_holidays = [  # Budapest
    (['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020', '03/27/2016', '04/16/2017', '04/01/2018', '04/21/2019'], 'Easter Day'), 
    (['04/01/2024', '04/10/2023', '04/18/2022', '04/05/2021', '04/13/2020', '03/28/2016', '04/17/2017', '04/02/2018', '04/22/2019'], 'Easter Monday'),
    (['05/12/2024', '05/10/2020', '05/09/2021', '05/08/2022', '05/14/2023', '05/08/2016', '05/14/2017', '05/13/2018', '05/12/2019'], "Mother's Day"),
    (['03/25/2016', '04/14/2017', '03/30/2018', '04/20/2019', '04/11/2020', '04/03/2021', '04/17/2022', '04/08/2023'], 'Good Friday'),
]

# Bavaria - Munich
munich_holidays = [
    (['03/30/2024', '04/08/2023', '04/16/2022', '04/03/2021', '03/26/2016', '04/15/2017', '03/31/2018', '04/20/2019'], 'Holy Saturday'), 
    (['05/12/2024', '05/14/2023', '05/08/2022', '05/09/2021', '05/08/2016', '05/14/2017', '05/13/2018', '05/12/2019'], "Mother's Day"), 
    (['03/25/2016', '04/14/2017', '03/30/2018', '04/20/2019', '04/11/2020', '04/03/2021', '04/16/2022', '04/08/2023'], 'Good Friday'),
    (['03/27/2016', '04/16/2017', '04/01/2018', '04/21/2019', '04/12/2020', '04/04/2021', '04/17/2022', '04/09/2023'], 'Easter Sunday'),
    (['03/28/2016', '04/17/2017', '04/02/2018', '04/22/2019', '04/13/2020', '04/05/2021', '04/18/2022', '04/10/2023'], 'Easter Monday'),
]

# Hesse - Frankfurt
frank_holidays = [
    (['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020', '03/27/2016', '04/16/2017', '04/01/2018', '04/21/2019'], 'Easter Day'),
    (['04/01/2024', '04/10/2023', '04/18/2022', '04/05/2021', '04/13/2020', '03/28/2016', '04/17/2017', '04/02/2018', '04/22/2019'], 'Easter Monday'),
    (['03/30/2024', '04/08/2023', '04/16/2022', '04/03/2021', '03/26/2016', '04/15/2017', '03/31/2018', '04/20/2019'], 'Holy Saturday'), 
    (['05/12/2024', '05/14/2023', '05/08/2022', '05/09/2021', '05/08/2016', '05/14/2017', '05/13/2018', '05/12/2019'], "Mother's Day"),
    (['03/25/2016', '04/14/2017', '03/30/2018', '04/20/2019', '04/11/2020', '04/03/2021', '04/16/2022', '04/08/2023'], 'Good Friday'),
    (['03/27/2016', '04/16/2017', '04/01/2018', '04/21/2019', '04/12/2020', '04/04/2021', '04/17/2022', '04/09/2023'], 'Easter Sunday'),
]

# df_fill function
def fill_loss_holidays(df_fill, warehouses, holidays):
    df = df_fill.copy()
    for item in holidays: 
        dates, holiday_name = item
        # 对日期进行格式化操作 12/29/2019 会变成 2019-12-29
        generated_dates = [datetime.strptime(date, '%m/%d/%Y').strftime('%Y-%m-%d') for date in dates]
        for generated_date in generated_dates:
            df.loc[(df['warehouse'].isin(warehouses)) & (df['date'] == generated_date), 'holiday'] = 1
            df.loc[(df['warehouse'].isin(warehouses)) & (df['date'] == generated_date), 'holiday_name'] = holiday_name
    return df

# 给每家店添加节日
calendar = fill_loss_holidays(df_fill=calendar, warehouses=['Prague_1', 'Prague_2', 'Prague_3'], holidays=czech_holiday)
calendar = fill_loss_holidays(df_fill=calendar, warehouses=['Brno_1'], holidays=brno_holiday)
calendar = fill_loss_holidays(df_fill=calendar, warehouses=['Munich_1'], holidays=munich_holidays)
calendar = fill_loss_holidays(df_fill=calendar, warehouses=['Frankfurt_1'], holidays=frank_holidays)
calendar = fill_loss_holidays(df_fill=calendar, warehouses=['Budapest_1'], holidays=budapest_holidays)

# 把有假日名字但没标holiday的补上
calendar.loc[calendar['holiday_name'].notna(), 'holiday'] = 1

# 每家店按照日期排序
calendar = calendar.sort_values(by=['warehouse', 'date'])
calendar.head()


,warehouse,date,holiday,holiday_name
17698,Brno_1,2016-01-01,1,New Years Day
12672,Brno_1,2016-01-02,0,NaN
12440,Brno_1,2016-01-03,0,NaN
7344,Brno_1,2016-01-04,0,NaN
11523,Brno_1,2016-01-05,0,NaN


## 分出年月日星期列

In [9]:
# create new columns for year, month, and day of week based on date
calendar['year'] = calendar['date'].dt.year
calendar['month'] = calendar['date'].dt.month
calendar['day'] = calendar['date'].dt.day
calendar['day_of_week'] = calendar['date'].dt.dayofweek

In [10]:
#验证：所有节日都有名字（并没有，交给小智改）
print(calendar[(calendar["holiday"] == 1) & (pd.isna(calendar["holiday_name"]))])

        warehouse       date  holiday holiday_name  year  month  day  \
22231      Brno_1 2016-03-26        1          NaN  2016      3   26   
15136      Brno_1 2017-04-15        1          NaN  2017      4   15   
22235      Brno_1 2018-03-31        1          NaN  2018      3   31   
22234      Brno_1 2024-03-30        1          NaN  2024      3   30   
12881  Budapest_1 2016-03-26        1          NaN  2016      3   26   
18110  Budapest_1 2017-04-15        1          NaN  2017      4   15   
7671   Budapest_1 2018-03-31        1          NaN  2018      3   31   
2453   Budapest_1 2024-03-30        1          NaN  2024      3   30   
463      Munich_1 2024-03-31        1          NaN  2024      3   31   
22229    Prague_1 2016-03-26        1          NaN  2016      3   26   
22236    Prague_1 2017-04-15        1          NaN  2017      4   15   
4699     Prague_1 2018-03-31        1          NaN  2018      3   31   
9928     Prague_1 2024-03-30        1          NaN  2024      3 

## 添加在某一年出现但是其他年份没有被标记的节日

In [5]:
# for each warehouse for each holiday, check if next year has the same holiday on the same day. If not, change mark the same day in the next year as holiday and change its holida_name to the same as this year. Do this for years except the last year.
for warehouse in calendar['warehouse'].unique():
    for holiday_name in calendar.loc[calendar['warehouse'] == warehouse, 'holiday_name'].unique():
        for year in calendar.loc[calendar['warehouse'] == warehouse, 'year'].unique():
            if year != calendar.loc[calendar['warehouse'] == warehouse, 'year'].max():
                holiday_date = calendar.loc[(calendar['warehouse'] == warehouse) & (calendar['year'] == year) & (calendar['holiday_name'] == holiday_name), 'date'].values[0]
                next_year = year + 1
                next_year_holiday_date = holiday_date + 1
                if next_year_holiday_date not in calendar.loc[(calendar['warehouse'] == warehouse) & (calendar['year'] == next_year), 'date'].values:
                    calendar.loc[(calendar['warehouse'] == warehouse) & (calendar['date'] == next_year_holiday_date), 'holiday'] = 1
                    calendar.loc[(calendar['warehouse'] == warehouse) & (calendar['date'] == next_year_holiday_date), 'holiday_name'] = holiday_name

KeyError: 'year'

In [31]:
print(calendar)

      warehouse       date  holiday   holiday_name  year  month  day  \
17698    Brno_1 2016-01-01        1  New Years Day  2016      1    1   
12672    Brno_1 2016-01-02        0            NaN  2016      1    2   
12440    Brno_1 2016-01-03        0            NaN  2016      1    3   
7344     Brno_1 2016-01-04        0            NaN  2016      1    4   
11523    Brno_1 2016-01-05        0            NaN  2016      1    5   
...         ...        ...      ...            ...   ...    ...  ...   
7476   Prague_3 2024-12-27        0            NaN  2024     12   27   
4655   Prague_3 2024-12-28        0            NaN  2024     12   28   
22168  Prague_3 2024-12-29        0            NaN  2024     12   29   
12542  Prague_3 2024-12-30        0            NaN  2024     12   30   
3915   Prague_3 2024-12-31        0            NaN  2024     12   31   

       day_of_week  
17698            4  
12672            5  
12440            6  
7344             0  
11523            1  
...      

## cld data cleaning basics
√小写所有字母
√去名字空格
√每个节日有单独column

In [32]:
# Step 1: Clean the `holiday_name` column
calendar["holiday_name"] = calendar["holiday_name"].str.lower()  # Convert to lowercase
calendar["holiday_name"] = calendar["holiday_name"].str.replace(" ", "_")  # Remove spaces

# Step 2: Get unique holiday names (excluding None)
unique_holidays = calendar["holiday_name"].dropna().unique()

# Step 3: Create a separate column for each holiday based on the 'holiday' column
for hld in unique_holidays:
    hld_column = []

    # Loop through each row in the 'holiday_name' column
    for index, row in calendar.iterrows():
        # Check if the current row's holiday_name matches the holiday, and if the holiday column is 1
        if row['holiday_name'] == hld and row['holiday'] == 1:
            hld_column.append(1)  # Mark as holiday
        else:
            hld_column.append(0)  # Mark as not a holiday

    # Assign the new column to the DataFrame
    calendar[hld] = hld_column

print(calendar.columns)

Index(['warehouse', 'date', 'holiday', 'holiday_name', 'year', 'month', 'day',
       'day_of_week', 'new_years_day', 'international_womens_day',
       'good_friday', 'easter_day', 'easter_monday', 'labour_day',
       'mother's_day', 'cyrila_a_metodej', 'jan_hus', 'den_ceske_statnosti',
       'den_vzniku_samostatneho_ceskoslovenskeho_statu',
       'den_boje_za_svobodu_a_demokracii', 'christmas_eve',
       '1st_christmas_day', '2nd_christmas_day', 'den_osvobozeni',
       'memorial_day_of_the_republic',
       'memorial_day_for_the_victims_of_the_communist_dictatorships',
       'memorial_day_for_the_victims_of_the_holocaust', 'whit_sunday',
       'whit_monday', 'national_defense_day', 'day_of_national_unity',
       'independent_hungary_day', 'state_foundation_day',
       'memorial_day_for_the_martyrs_of_arad',
       'memorial_day_of_the_1956_revolution', 'all_saints_day',
       'hungary_national_day_holiday', 'christmas_holiday',
       '1848_revolution_memorial_day_(extra_ho

## 节日前后考量
节日前后也标记为holiday
如果节日放
加入boolean columns day_before_holiday 和 day_after_holiday

In [33]:
def fill_calendar2df(calendar, df_cld):
    # df_cld已经是holiday=1的数据了
    for _, row in df_cld.iterrows():
        # 店,节日名,节日的日期
        warehouse, holiday_date, holiday_name = row['warehouse'], row['date'], row['holiday_name']
        ## 劳动节和复活节是特殊节日, date_range为[-2,1],普通节日是[-1,1]
        # 在英国，复活节假期通常持续四天,欧洲劳动节貌似是1天+周末放假
        if holiday_name in ['Labour Day', 'Easter Monday']:
            date_range = pd.date_range(start=holiday_date - pd.Timedelta(days=2), end=holiday_date + pd.Timedelta(days=1))
        else:  # 可能是调休,节日放假1天+周末的放假
            date_range = pd.date_range(start=holiday_date - pd.Timedelta(days=1), end=holiday_date + pd.Timedelta(days=1))
        # date_range:[-2,-1,0,1]
        for i, date in enumerate(date_range):
            mask = (calendar['warehouse'] == warehouse) & (calendar['date'] == date)
            calendar.loc[mask, 'holiday'] = 1
            # 如果不是最后一天(也就是date_range里的1),就算作holiday
            if i + 1 != len(date_range):
                calendar.loc[mask, 'holiday_name'] = holiday_name
    return calendar

# snow和precipitation由于测试集中没有,后面会drop,缺失值列就只有holiday_name了,对holiday_name进行填充‘Not’
calendar = calendar.fillna('Not')
# 转成float是为了后面特殊节日的1.5
calendar['holiday'] = calendar['holiday'].astype(float)
calendar = fill_calendar2df(calendar, calendar)
calendar.head()

# 复活节日期列表
datesx = ['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020']
# 时间字符串格式化得到复活节前1天
holidaysx = [datetime.strptime(date, '%m/%d/%Y') - timedelta(days=1) for date in datesx]
# 这3家店复活节前1天,标记为holiday
warehouses = ['Prague_1', 'Prague_2', 'Prague_3']
calendar.loc[(calendar['date'].isin(holidaysx)) & (calendar['warehouse'].isin(warehouses)), 'holiday'] = 1

# 构造一天前是不是holiday,一天后是不是holiday的特征,fillna(-1)和普通的0进行区别
calendar['day_before_holiday'] = calendar['holiday'].shift(-1).fillna(-1)
calendar['day_after_holiday'] = calendar['holiday'].shift().fillna(-1)

#拜拜了您内
calendar = calendar.drop(columns=['holiday'])

KeyboardInterrupt: 

In [ ]:
#检查column名字数量都没问题
print(calendar.columns)

Index(['warehouse', 'date', 'holiday_name', 'year', 'month', 'day',
       'day_of_week', 'new_years_day', 'international_womens_day',
       'labour_day', 'den_osvobozeni', 'cyrila_a_metodej', 'jan_hus',
       'den_ceske_statnosti', 'den_vzniku_samostatneho_ceskoslovenskeho_statu',
       'den_boje_za_svobodu_a_demokracii', 'christmas_eve',
       '1st_christmas_day', '2nd_christmas_day', 'good_friday',
       'easter_monday', 'easter_day', 'mother_day',
       'memorial_day_of_the_republic',
       'memorial_day_for_the_victims_of_the_communist_dictatorships',
       'memorial_day_for_the_victims_of_the_holocaust', 'whit_sunday',
       'whit_monday', 'national_defense_day', 'day_of_national_unity',
       'independent_hungary_day', 'state_foundation_day',
       'memorial_day_for_the_martyrs_of_arad',
       'memorial_day_of_the_1956_revolution', 'all_saints_day',
       'hungary_national_day_holiday', 'christmas_holiday',
       '1848_revolution_memorial_day_(extra_holiday)',
    

## 与主表合并

In [ ]:
#检查有无缺少的日期
start_date = calendar['date'].min()
end_date = calendar['date'].max()
print(start_date, end_date)
expected_dates = pd.date_range(start=start_date, end=end_date)
missing_dates = expected_dates.difference(calendar['date'])
if missing_dates.empty:
    print("The calendar contains all dates within the given range.")
else:
    print("Missing dates in the calendar:")
    print(missing_dates)


2016-01-01 00:00:00 2024-12-31 00:00:00
The calendar contains all dates within the given range.


In [ ]:
sales_train = pd.read_csv('sales_train.csv',parse_dates=['date'])
sales_test = pd.read_csv('sales_test.csv',parse_dates=['date'])
inventory = pd.read_csv('inventory.csv')

In [ ]:
#合并 sales train, sales test与inventory
sales_train = pd.merge(sales_train, inventory, how='left', on =['unique_id','warehouse'])
sales_test = pd.merge(sales_test, inventory, how='left', on =['unique_id','warehouse'])
#sales_train
sales_test

,unique_id,date,warehouse,total_orders,sell_price_main,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount,product_unique_id,name,L1_category_name_en,L2_category_name_en,L3_category_name_en,L4_category_name_en
0,1226,2024-06-03,Brno_1,8679.0,13.13,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,627,Croissant_9,Bakery,Bakery_L2_14,Bakery_L3_37,Bakery_L4_1
1,1226,2024-06-11,Brno_1,8795.0,13.13,0.15873,0.0,0.0,0.0,0.0,0.0,0.0,627,Croissant_9,Bakery,Bakery_L2_14,Bakery_L3_37,Bakery_L4_1
2,1226,2024-06-13,Brno_1,10009.0,13.13,0.15873,0.0,0.0,0.0,0.0,0.0,0.0,627,Croissant_9,Bakery,Bakery_L2_14,Bakery_L3_37,Bakery_L4_1
3,1226,2024-06-15,Brno_1,8482.0,13.13,0.15873,0.0,0.0,0.0,0.0,0.0,0.0,627,Croissant_9,Bakery,Bakery_L2_14,Bakery_L3_37,Bakery_L4_1
4,1226,2024-06-09,Brno_1,8195.0,13.13,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,627,Croissant_9,Bakery,Bakery_L2_14,Bakery_L3_37,Bakery_L4_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47016,4572,2024-06-03,Munich_1,5254.0,2.09,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,2245,Apple_123,Fruit and vegetable,Fruit and vegetable_L2_1,Fruit and vegetable_L3_31,Fruit and vegetable_L4_1
47017,3735,2024-06-04,Prague_1,9698.0,11.00,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1833,Kiwi_18,Fruit and vegetable,Fruit and vegetable_L2_1,Fruit and vegetable_L3_39,Fruit and vegetable_L4_52
47018,3735,2024-06-03,Prague_1,10256.0,11.00,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1833,Kiwi_18,Fruit and vegetable,Fruit and vegetable_L2_1,Fruit and vegetable_L3_39,Fruit and vegetable_L4_52
47019,2129,2024-06-03,Brno_1,8679.0,37.75,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1074,Grape_15,Fruit and vegetable,Fruit and vegetable_L2_1,Fruit and vegetable_L3_12,Fruit and vegetable_L4_1


In [ ]:
#合并 sales train, sales test与calendar
sales_train = pd.merge(sales_train, calendar, how='left', on=['date', 'warehouse'])
sales_test = pd.merge(sales_test, calendar, how='left', on=['date', 'warehouse'])
sales_test

,unique_id,date,warehouse,total_orders,sell_price_main,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,...,ascension_day,corpus_christi,german_unity_day,reformation_day,holy_saturday,epiphany,assumption_of_the_virgin_mary,peace_festival_in_augsburg,day_before_holiday,day_after_holiday
0,1226,2024-06-03,Brno_1,8679.0,13.13,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
1,1226,2024-06-11,Brno_1,8795.0,13.13,0.15873,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
2,1226,2024-06-13,Brno_1,10009.0,13.13,0.15873,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
3,1226,2024-06-15,Brno_1,8482.0,13.13,0.15873,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
4,1226,2024-06-09,Brno_1,8195.0,13.13,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47016,4572,2024-06-03,Munich_1,5254.0,2.09,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
47017,3735,2024-06-04,Prague_1,9698.0,11.00,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
47018,3735,2024-06-03,Prague_1,10256.0,11.00,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
47019,2129,2024-06-03,Brno_1,8679.0,37.75,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0


In [ ]:
#sales_train里有但是sales_test里没有的列，包含一些节日和availability
np.setdiff1d(sales_train.columns, sales_test.columns)

array(['availability', 'sales'], dtype=object)

In [ ]:
# Print columns in sales_train
print("Columns in sales_train:")
print(sales_train.columns.tolist())

# Print columns in sales_test
print("\nColumns in sales_test:")
print(sales_test.columns.tolist())

# Compare the columns to find differences
columns_only_in_train = set(sales_train.columns) - set(sales_test.columns)
columns_only_in_test = set(sales_test.columns) - set(sales_train.columns)

print("\nColumns present only in sales_train:", columns_only_in_train)
print("Columns present only in sales_test:", columns_only_in_test)

# Check if columns are identical
if columns_only_in_train or columns_only_in_test:
    print("\nThe columns are not identical between sales_train and sales_test.")
else:
    print("\nThe columns are identical between sales_train and sales_test.")


Columns in sales_train:
['unique_id', 'date', 'warehouse', 'total_orders', 'sales', 'sell_price_main', 'availability', 'type_0_discount', 'type_1_discount', 'type_2_discount', 'type_3_discount', 'type_4_discount', 'type_5_discount', 'type_6_discount', 'product_unique_id', 'name', 'L1_category_name_en', 'L2_category_name_en', 'L3_category_name_en', 'L4_category_name_en', 'holiday_name', 'year', 'month', 'day', 'day_of_week', 'new_years_day', 'international_womens_day', 'labour_day', 'den_osvobozeni', 'cyrila_a_metodej', 'jan_hus', 'den_ceske_statnosti', 'den_vzniku_samostatneho_ceskoslovenskeho_statu', 'den_boje_za_svobodu_a_demokracii', 'christmas_eve', '1st_christmas_day', '2nd_christmas_day', 'good_friday', 'easter_monday', 'easter_day', 'mother_day', 'memorial_day_of_the_republic', 'memorial_day_for_the_victims_of_the_communist_dictatorships', 'memorial_day_for_the_victims_of_the_holocaust', 'whit_sunday', 'whit_monday', 'national_defense_day', 'day_of_national_unity', 'independent_

In [ ]:
# drop 'availablity' in sales_train
sales_train.drop(['availability'], axis=1, inplace=True)

In [ ]:
sales_train.sort_values(['date', 'warehouse'], inplace=True)
sales_train

,unique_id,date,warehouse,total_orders,sales,sell_price_main,type_0_discount,type_1_discount,type_2_discount,type_3_discount,...,ascension_day,corpus_christi,german_unity_day,reformation_day,holy_saturday,epiphany,assumption_of_the_virgin_mary,peace_festival_in_augsburg,day_before_holiday,day_after_holiday
5645,2706,2020-08-01,Brno_1,4797.0,40.07,56.90,0.00000,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
22442,5033,2020-08-01,Brno_1,4797.0,17.81,45.09,0.00000,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
30684,3285,2020-08-01,Brno_1,4797.0,40.07,17.84,0.00000,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
36380,1807,2020-08-01,Brno_1,4797.0,17.81,92.78,0.00000,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
41183,3200,2020-08-01,Brno_1,4797.0,251.54,28.85,0.00000,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3997343,362,2024-06-02,Prague_3,5177.0,128.58,61.90,0.00000,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
3998329,3907,2024-06-02,Prague_3,5177.0,69.37,93.18,0.00000,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
4000870,1345,2024-06-02,Prague_3,5392.0,86.98,14.49,0.00000,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
4001389,4747,2024-06-02,Prague_3,5177.0,389.12,275.44,0.06972,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0


In [ ]:
for df in [sales_train, sales_test]:
    df.set_index('date', inplace=True)

In [ ]:
sales_train

,unique_id,warehouse,total_orders,sales,sell_price_main,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,...,ascension_day,corpus_christi,german_unity_day,reformation_day,holy_saturday,epiphany,assumption_of_the_virgin_mary,peace_festival_in_augsburg,day_before_holiday,day_after_holiday
date,,,,,,,,,,,,,,,,,,,,,
2020-08-01,2706,Brno_1,4797.0,40.07,56.90,0.00000,0.0,0.00000,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
2020-08-01,5033,Brno_1,4797.0,17.81,45.09,0.00000,0.0,0.00000,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
2020-08-01,3285,Brno_1,4797.0,40.07,17.84,0.00000,0.0,0.00000,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
2020-08-01,1807,Brno_1,4797.0,17.81,92.78,0.00000,0.0,0.00000,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
2020-08-01,3200,Brno_1,4797.0,251.54,28.85,0.00000,0.0,0.00000,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-06-02,362,Prague_3,5177.0,128.58,61.90,0.00000,0.0,0.00000,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
2024-06-02,3907,Prague_3,5177.0,69.37,93.18,0.00000,0.0,0.00000,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
2024-06-02,1345,Prague_3,5392.0,86.98,14.49,0.00000,0.0,0.00000,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
